# Descricao analitica das bases de dados

Resume a camada `processed/textos_parlamentares/v1` por fonte, dataset, ano, familia textual e disponibilidade de campos. Use depois de rodar a normalizacao.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
PROCESSED_VERSION = 'v1'
PROCESSED_ROOT = DATA_ROOT / 'processed' / 'textos_parlamentares' / PROCESSED_VERSION
MANIFEST_ROOT = DATA_ROOT / 'processed' / 'manifests'
RAW_MANIFEST_ROOT = DATA_ROOT / 'manifests'

os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)


## Selecionar execucao processada

Por padrao, usa o manifest processed mais recente. Para fixar uma execucao, preencha `MANIFEST_PATH`.


In [ ]:
import json
from pathlib import Path

MANIFEST_PATH = None  # exemplo: DATA_ROOT / 'processed' / 'manifests' / 'processed-textos-v1-20260522.json'

if MANIFEST_PATH is None:
    manifests = sorted(MANIFEST_ROOT.glob('*.json'), key=lambda path: path.stat().st_mtime, reverse=True)
    if not manifests:
        raise FileNotFoundError(f'Nenhum manifest processed encontrado em {MANIFEST_ROOT}')
    MANIFEST_PATH = manifests[0]

manifest = json.loads(Path(MANIFEST_PATH).read_text(encoding='utf-8'))
output_files = [Path(path) for path in manifest.get('output_files', [])]

print('Manifest selecionado:', MANIFEST_PATH)
print('RUN_ID:', manifest.get('run_id'))
print('Arquivos:', len(output_files))
print('Registros:', manifest.get('output_records'))


## Carregar registros

Carrega somente os campos necessarios para descricao. Se a base estiver muito grande, ajuste `MAX_ROWS` para uma leitura parcial exploratoria.


In [ ]:
import json
from pathlib import Path

MAX_ROWS = None
rows = []

for path in output_files:
    if MAX_ROWS is not None and len(rows) >= MAX_ROWS:
        break
    if not path.exists():
        print('Arquivo ausente:', path)
        continue
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if MAX_ROWS is not None and len(rows) >= MAX_ROWS:
                break
            row = json.loads(line)
            rows.append(row)

print('Registros carregados:', len(rows))


## Cobertura geral

Contagens por fonte, dataset, familia textual e ano.


In [ ]:
from collections import Counter, defaultdict

def key(row, *fields):
    return tuple(row.get(field) for field in fields)

def print_counter(title, counter, limit=50):
    print(title)
    for item, count in counter.most_common(limit):
        print(count, item)
    print()

print_counter('Por source/dataset/documento_tipo:', Counter(key(row, 'source', 'dataset', 'documento_tipo') for row in rows))
print_counter('Por ano/documento_tipo:', Counter(key(row, 'ano', 'documento_tipo') for row in rows))
print_counter('Por ambito:', Counter(row.get('ambito') for row in rows))


In [ ]:
coverage = defaultdict(lambda: {'registros': 0, 'anos': set(), 'texto_total': 0})
for row in rows:
    group = (row.get('source'), row.get('dataset'), row.get('documento_tipo'))
    coverage[group]['registros'] += 1
    coverage[group]['anos'].add(row.get('ano'))
    coverage[group]['texto_total'] += row.get('texto_tamanho') or 0

for group, stats in sorted(coverage.items()):
    anos = sorted(year for year in stats['anos'] if year)
    media = stats['texto_total'] / stats['registros'] if stats['registros'] else 0
    print({
        'base': group,
        'registros': stats['registros'],
        'anos': f"{anos[0]}-{anos[-1]}" if anos else None,
        'texto_medio_caracteres': round(media, 1),
    })


## Qualidade dos campos

Mostra preenchimento dos campos mais importantes para analise.


In [ ]:
FIELDS = [
    'data',
    'parlamentar_nome',
    'proposicao_identificacao',
    'documento_classe',
    'status_deliberativo',
    'url_texto',
    'url_video',
    'raw_path',
]

groups = defaultdict(list)
for row in rows:
    groups[(row.get('source'), row.get('dataset'), row.get('documento_tipo'))].append(row)

for group, items in sorted(groups.items()):
    print()
    print('BASE:', group, 'registros:', len(items))
    for field in FIELDS:
        filled = sum(1 for item in items if item.get(field) not in (None, '', []))
        pct = 100 * filled / len(items) if items else 0
        print(f'  {field}: {filled}/{len(items)} ({pct:.1f}%)')


## Exemplos por base

Imprime um registro compacto de cada base para revisao humana.


In [ ]:
import json

for group, items in sorted(groups.items()):
    row = items[0]
    print()
    print('BASE:', group)
    print(json.dumps({
        'texto_id': row.get('texto_id'),
        'casa': row.get('casa'),
        'ambito': row.get('ambito'),
        'data': row.get('data'),
        'titulo': row.get('titulo'),
        'parlamentar_nome': row.get('parlamentar_nome'),
        'proposicao_identificacao': row.get('proposicao_identificacao'),
        'documento_classe': row.get('documento_classe'),
        'status_deliberativo': row.get('status_deliberativo'),
        'texto_tamanho': row.get('texto_tamanho'),
        'raw_path': row.get('raw_path'),
    }, ensure_ascii=False, indent=2))


## Manifests de coleta relacionados

Lista os manifests brutos que alimentaram a execucao processada.


In [ ]:
raw_run_ids = manifest.get('raw_run_ids', [])
for run_id in raw_run_ids:
    path = RAW_MANIFEST_ROOT / f'{run_id}.json'
    if not path.exists():
        continue
    raw_manifest = json.loads(path.read_text(encoding='utf-8'))
    print({
        'run_id': run_id,
        'source': raw_manifest.get('source'),
        'dataset': raw_manifest.get('dataset'),
        'status': raw_manifest.get('status'),
        'record_counts': raw_manifest.get('record_counts'),
    })
